In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE paysim AS
SELECT *
FROM read_csv_auto('../data/paysim_transactions.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
row_count = con.execute("""
SELECT COUNT(*) AS total_rows
FROM paysim;
""").df()

row_count

,total_rows
0,6362620


In [4]:
con.execute("""
SELECT
    type,
    isFraud,
    COUNT(*) AS transaction_count,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(AVG(oldbalanceOrg), 2) AS avg_oldbalance_origin,
    ROUND(AVG(newbalanceOrig), 2) AS avg_newbalance_origin
FROM paysim
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY type, isFraud
ORDER BY type, isFraud DESC;
""").df()

,type,isFraud,transaction_count,avg_amount,avg_oldbalance_origin,avg_newbalance_origin
0,CASH_OUT,1,4116,1455102.59,1453869.05,72.59
1,CASH_OUT,0,2233384,173917.16,43429.23,17506.26
2,TRANSFER,1,4097,1480891.67,1846374.19,385604.57
3,TRANSFER,0,528812,906229.01,40558.76,7380.37


In [5]:
con.execute("""
SELECT
    type,
    isFraud,
    COUNT(*) AS transaction_count,
    SUM(CASE WHEN newbalanceOrig = 0 THEN 1 ELSE 0 END) AS zero_origin_balance_count,
    ROUND(
        SUM(CASE WHEN newbalanceOrig = 0 THEN 1 ELSE 0 END) * 1.0 / COUNT(*),
        4
    ) AS zero_origin_balance_rate
FROM paysim
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY type, isFraud
ORDER BY type, isFraud DESC;
""").df()

,type,isFraud,transaction_count,zero_origin_balance_count,zero_origin_balance_rate
0,CASH_OUT,1,4116,4115.0,0.9998
1,CASH_OUT,0,2233384,1981096.0,0.8870
2,TRANSFER,1,4097,3938.0,0.9612
3,TRANSFER,0,528812,507507.0,0.9597


In [6]:
con.execute("""
SELECT
    type,
    isFraud,
    COUNT(*) AS transaction_count,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(AVG(oldbalanceOrg), 2) AS avg_oldbalance_origin,
    ROUND(AVG(amount / NULLIF(oldbalanceOrg, 0)), 4) AS avg_depletion_ratio
FROM paysim
WHERE type IN ('TRANSFER', 'CASH_OUT')
  AND oldbalanceOrg > 0
GROUP BY type, isFraud
ORDER BY type, isFraud DESC;
""").df()

,type,isFraud,transaction_count,avg_amount,avg_oldbalance_origin,avg_depletion_ratio
0,CASH_OUT,1,4079,1467094.37,1467056.88,1.0122
1,CASH_OUT,0,1207622,178369.73,80318.29,178.4423
2,TRANSFER,1,4093,1481212.42,1848178.61,0.9854
3,TRANSFER,0,246033,876760.33,87175.13,762.9657


In [7]:
con.execute("""
SELECT
    type,
    isFraud,
    COUNT(*) AS transaction_count,
    SUM(
        CASE
            WHEN oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1
            ELSE 0
        END
    ) AS high_depletion_count,
    ROUND(
        SUM(
            CASE
                WHEN oldbalanceOrg > 0
                     AND amount >= oldbalanceOrg * 0.90
                THEN 1
                ELSE 0
            END
        ) * 1.0 / COUNT(*),
        4
    ) AS high_depletion_rate
FROM paysim
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY type, isFraud
ORDER BY type, isFraud DESC;
""").df()

,type,isFraud,transaction_count,high_depletion_count,high_depletion_rate
0,CASH_OUT,1,4116,4078.0,0.9908
1,CASH_OUT,0,2233384,974099.0,0.4362
2,TRANSFER,1,4097,3958.0,0.9661
3,TRANSFER,0,528812,226656.0,0.4286


In [8]:
con.execute("""
WITH rule_results AS (
    SELECT
        type,
        amount,
        oldbalanceOrg,
        isFraud,
        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount >= 500000
                 AND oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim
),

rule_summary AS (
    SELECT
        COUNT(*) AS total_transactions,
        SUM(rule_flag) AS flagged_transactions,
        SUM(isFraud) AS total_fraud_transactions,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
        SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives
    FROM rule_results
)

SELECT
    total_transactions,
    flagged_transactions,
    total_fraud_transactions,
    true_positives,
    false_positives,
    false_negatives,
    ROUND(true_positives * 1.0 / total_fraud_transactions, 4) AS recall,
    ROUND(true_positives * 1.0 / flagged_transactions, 4) AS precision
FROM rule_summary;
""").df()

,total_transactions,flagged_transactions,total_fraud_transactions,true_positives,false_positives,false_negatives,recall,precision
0,6362620,158363.0,8213.0,3725.0,154638.0,4488.0,0.4535,0.0235


In [9]:
improved_rule_results = con.execute("""
WITH rule_results AS (
    SELECT
        type,
        amount,
        oldbalanceOrg,
        isFraud,
        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount >= 500000
                 AND oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim
),

rule_summary AS (
    SELECT
        COUNT(*) AS total_transactions,
        SUM(rule_flag) AS flagged_transactions,
        SUM(isFraud) AS total_fraud_transactions,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
        SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives
    FROM rule_results
)

SELECT
    total_transactions,
    flagged_transactions,
    total_fraud_transactions,
    true_positives,
    false_positives,
    false_negatives,
    ROUND(true_positives * 1.0 / total_fraud_transactions, 4) AS recall,
    ROUND(true_positives * 1.0 / flagged_transactions, 4) AS precision
FROM rule_summary;
""").df()

improved_rule_results.to_csv("../outputs/improved_fraud_rule_results.csv", index=False)

improved_rule_results

,total_transactions,flagged_transactions,total_fraud_transactions,true_positives,false_positives,false_negatives,recall,precision
0,6362620,158363.0,8213.0,3725.0,154638.0,4488.0,0.4535,0.0235
